In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR, SVC
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.naive_bayes import GaussianNB
import joblib
from tqdm import tqdm
import requests

In [4]:
#export LIBOMP_PATH=$(brew --prefix libomp)/lib
#export DYLD_LIBRARY_PATH=$LIBOMP_PATH:$DYLD_LIBRARY_PATH
#введи в терминале с активной сессией для работы с xgboost

## Forecast

#### Data preparation

In [5]:
df = pd.read_csv('data/epi_r.csv')

In [6]:
df

,title,rating,calories,protein,fat,sodium,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
0,"Lentil, Apple, and Turkey Wrap",2.500,426.0,30.0,7.0,559.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,Boudin Blanc Terrine with Red Onion Confit,4.375,403.0,18.0,23.0,1439.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Potato and Fennel Soup Hodge,3.750,165.0,6.0,7.0,165.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Mahi-Mahi in Tomato Olive Sauce,5.000,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Spinach Noodle Casserole,3.125,547.0,20.0,32.0,452.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20047,Parmesan Puffs,3.125,28.0,2.0,2.0,64.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20048,Artichoke and Parmesan Risotto,4.375,671.0,22.0,28.0,583.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20049,Turkey Cream Puff Pie,4.375,563.0,31.0,38.0,652.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
20050,Snapper on Angel Hair with Citrus Cream,4.375,631.0,45.0,24.0,517.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Filter the columns: the less non-ingredient columns in your dataset the better. You will predict the rating or rating category using only the ingredients and nothing else.

In [7]:
pd.set_option('display.max_columns', None)

In [8]:
print(df.columns.tolist())

['title', 'rating', 'calories', 'protein', 'fat', 'sodium', '#cakeweek', '#wasteless', '22-minute meals', '3-ingredient recipes', '30 days of groceries', 'advance prep required', 'alabama', 'alaska', 'alcoholic', 'almond', 'amaretto', 'anchovy', 'anise', 'anniversary', 'anthony bourdain', 'aperitif', 'appetizer', 'apple', 'apple juice', 'apricot', 'arizona', 'artichoke', 'arugula', 'asian pear', 'asparagus', 'aspen', 'atlanta', 'australia', 'avocado', 'back to school', 'backyard bbq', 'bacon', 'bake', 'banana', 'barley', 'basil', 'bass', 'bastille day', 'bean', 'beef', 'beef rib', 'beef shank', 'beef tenderloin', 'beer', 'beet', 'bell pepper', 'berry', 'beverly hills', 'birthday', 'biscuit', 'bitters', 'blackberry', 'blender', 'blue cheese', 'blueberry', 'boil', 'bok choy', 'bon appétit', 'bon app��tit', 'boston', 'bourbon', 'braise', 'bran', 'brandy', 'bread', 'breadcrumbs', 'breakfast', 'brie', 'brine', 'brisket', 'broccoli', 'broccoli rabe', 'broil', 'brooklyn', 'brown rice', 'brown

In [9]:
ingredients = [
    "rating", "almond", "anchovy", "apple", "apple juice", "apricot", "artichoke", "arugula", "asian pear", "asparagus", "avocado",
    "bacon", "banana", "barley", "basil", "beef", "beef rib", "beef shank", "beef tenderloin", "beet", "bell pepper",
    "berry", "blackberry", "blue cheese", "blueberry", "bok choy", "broccoli", "broccoli rabe", "brussel sprout",
    "butter", "buttermilk", "butternut squash", "cabbage", "capers", "caraway", "cardamom", "carrot", "cashew",
    "cauliflower", "caviar", "celery", "cheese", "cherry", "chestnut", "chickpea", "chile", "chile pepper", "chili",
    "chive", "chocolate", "cilantro", "cinnamon", "citrus", "clam", "clove", "coconut", "cod", "coffee", "collard greens",
    "corn", "cornmeal", "cottage cheese", "crab", "cranberry", "cream cheese", "cucumber", "cumin", "currant", "curry",
    "dairy", "date", "dill", "egg", "egg nog", "eggplant", "endive", "feta", "fig", "fish", "flat bread", "fontina",
    "fruit", "fruit juice", "garlic", "ginger", "goat cheese", "gouda", "grape", "grapefruit", "green bean",
    "green onion/scallion", "ground beef", "ground lamb", "guava", "hazelnut", "honey", "honeydew", "horseradish",
    "hot pepper", "hummus", "jalapeño", "jam or jelly", "jerusalem artichoke", "jícama", "kale", "kumquat", "lamb",
    "lamb chop", "lamb shank", "lentil", "lettuce", "lima bean", "lime", "lime juice", "lingonberry", "lobster",
    "macadamia nut", "mango", "maple syrup", "marinade", "marscarpone", "marshmallow", "melon", "mint", "molasses",
    "monterey jack", "mozzarella", "mushroom", "mussel", "mustard", "mustard greens", "nut", "nutmeg", "oat", "oatmeal",
    "octopus", "okra", "olive", "onion", "orange", "orange juice", "oregano", "orzo", "oyster", "parmesan", "parsley",
    "parsnip", "pea", "peach", "peanut", "peanut butter", "pear", "pecan", "pepper", "persimmon", "pine nut",
    "pineapple", "pistachio", "plantain", "plum", "pomegranate", "pomegranate juice", "poppy", "pork", "pork chop",
    "pork rib", "pork tenderloin", "potato", "prune", "pumpkin", "quail", "quince", "quinoa", "rabbit", "radicchio",
    "radish", "raisin", "raspberry", "rhubarb", "rice", "ricotta", "root vegetable", "rosemary", "rutabaga", "rye",
    "saffron", "sage", "salmon", "salsa", "sardine", "sausage", "scallop", "seed", "sesame", "sesame oil", "shallot",
    "shellfish", "shrimp", "snapper", "sorbet", "sour cream", "sourdough", "soy", "soy sauce", "spinach", "squash",
    "squid", "strawberry", "sugar snap pea", "sweet potato/yam", "swordfish", "tilapia", "tofu", "tomatillo", "tomato",
    "tortillas", "tree nut", "turnip", "vanilla", "veal", "vegetable", "venison", "vinegar", "wasabi", "watercress",
    "watermelon", "wheat/gluten-free", "wild rice", "yogurt", "yuca", "zucchini"
]


In [10]:
df = df[ingredients]

In [11]:
df

,rating,almond,anchovy,apple,apple juice,apricot,artichoke,arugula,asian pear,asparagus,avocado,bacon,banana,barley,basil,beef,beef rib,beef shank,beef tenderloin,beet,bell pepper,berry,blackberry,blue cheese,blueberry,bok choy,broccoli,broccoli rabe,brussel sprout,butter,buttermilk,butternut squash,cabbage,capers,caraway,cardamom,carrot,cashew,cauliflower,caviar,celery,cheese,cherry,chestnut,chickpea,chile,chile pepper,chili,chive,chocolate,cilantro,cinnamon,citrus,clam,clove,coconut,cod,coffee,collard greens,corn,cornmeal,cottage cheese,crab,cranberry,cream cheese,cucumber,cumin,currant,curry,dairy,date,dill,egg,egg nog,eggplant,endive,feta,fig,fish,flat bread,fontina,fruit,fruit juice,garlic,ginger,goat cheese,gouda,grape,grapefruit,green bean,green onion/scallion,ground beef,ground lamb,guava,hazelnut,honey,honeydew,horseradish,hot pepper,hummus,jalapeño,jam or jelly,jerusalem artichoke,jícama,kale,kumquat,lamb,lamb chop,lamb shank,lentil,lettuce,lima bean,lime,lime juice,lingonberry,lobster,macadamia nut,mango,maple syrup,marinade,marscarpone,marshmallow,melon,mint,molasses,monterey jack,mozzarella,mushroom,mussel,mustard,mustard greens,nut,nutmeg,oat,oatmeal,octopus,okra,olive,onion,orange,orange juice,oregano,orzo,oyster,parmesan,parsley,parsnip,pea,peach,peanut,peanut butter,pear,pecan,pepper,persimmon,pine nut,pineapple,pistachio,plantain,plum,pomegranate,pomegranate juice,poppy,pork,pork chop,pork rib,pork tenderloin,potato,prune,pumpkin,quail,quince,quinoa,rabbit,radicchio,radish,raisin,raspberry,rhubarb,rice,ricotta,root vegetable,rosemary,rutabaga,rye,saffron,sage,salmon,salsa,sardine,sausage,scallop,seed,sesame,sesame oil,shallot,shellfish,shrimp,snapper,sorbet,sour cream,sourdough,soy,soy sauce,spinach,squash,squid,strawberry,sugar snap pea,sweet potato/yam,swordfish,tilapia,tofu,tomatillo,tomato,tortillas,tree nut,turnip,vanilla,veal,vegetable,venison,vinegar,wasabi,watercress,watermelon,wheat/gluten-free,wild rice,yogurt,yuca,zucchini
0,2.500,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.

In [12]:
X = df.drop('rating', axis=1)
y = df['rating']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

#### Regression

In [14]:
## baseline

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

fit_intercepts = [True, False]

for fit_intercept in fit_intercepts:
    lr = LinearRegression(fit_intercept=fit_intercept)
    scores = []
    best_score = -1
    
    for train_index, valid_index in kfold.split(X_train, y_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

        lr.fit(X_train_fold, y_train_fold)

        y_pred = lr.predict(X_valid_fold)

        score = np.sqrt(mean_squared_error(y_pred, y_valid_fold))
        scores.append(score)
    av_score = np.mean(scores)

    if av_score > best_score:
        best_score = av_score
        best_fit_intercept = fit_intercept

print(f'Best score: {best_score}')
print(f'Best fit_intercept: {best_fit_intercept}')


Best score: 499336172309.65656
Best fit_intercept: False


In [15]:
models = {
    'Linear Regression': (LinearRegression(), {'fit_intercept': [True, False]}),
    "Ridge Regression": (Ridge(), {"alpha": [0.1, 1.0, 10.0]}),
    "Decision Tree Regressor": (DecisionTreeRegressor(), {"max_depth" : [5, 10, 20]}),
    "Random Forest": (RandomForestRegressor(), {"n_estimators": [100, 200], "max_depth": [10, 20]}),
    "Gradient Boosting": (GradientBoostingRegressor(), {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1]}),
    "Support Vector Regressor": (SVR(), {"C": [0.1, 1.0]}),
}

for model_name, (model, params) in models.items():
    print(f'GridSearch for model {model}')
    
    grid_search = GridSearchCV(model, params, cv=3, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_pred = best_model.predict(X_test)
    rmse_score = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f'Best params: {best_params}')
    print(f'RMSE on Test: {rmse_score}')
    print()

GridSearch for model LinearRegression()
Best params: {'fit_intercept': True}
RMSE on Test: 1.3095292539239385

GridSearch for model Ridge()
Best params: {'alpha': 10.0}
RMSE on Test: 1.3096068244739059

GridSearch for model DecisionTreeRegressor()
Best params: {'max_depth': 5}
RMSE on Test: 1.3312394854517975

GridSearch for model RandomForestRegressor()
Best params: {'max_depth': 20, 'n_estimators': 200}
RMSE on Test: 1.3108521365994237

GridSearch for model GradientBoostingRegressor()
Best params: {'learning_rate': 0.1, 'n_estimators': 200}
RMSE on Test: 1.3075374260633212

GridSearch for model SVR()
Best params: {'C': 1.0}
RMSE on Test: 1.355456047048189



In [16]:
## Naive Regressor
mean_rating = np.mean(df['rating'])
y_pred = np.full_like(y_test, mean_rating)
score = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'RMSE on Test for Naive Regressor: {score}')

RMSE on Test for Naive Regressor: 1.340604734722414


#### Classification

In [17]:
binarized_y = y.round()

X_train, X_test, y_train, y_test = train_test_split(X, binarized_y, test_size=0.2, random_state=42, stratify=binarized_y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) 

In [18]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [19]:
models = {
    "Logistic Regression": (LogisticRegression(max_iter=5000), {"C": [0.1, 1]}),
    "Decision Tree": (DecisionTreeClassifier(), {"max_depth": [5, 10, 20]}),
    "Random Forest": (RandomForestClassifier(), {"n_estimators": [100, 200], "max_depth": [10, 20]}),
    "Gradient Boosting": (GradientBoostingClassifier(), {"n_estimators": [50, 100], "learning_rate": [0.05, 0.1]}),
    "Support Vector Classifier": (SVC(), {"C": [0.1, 1.0]})
}
for model_name, (model, params) in models.items():
    print(f"GridSearch for {model_name}")

    grid_search = GridSearchCV(model, params, cv=skf, scoring="accuracy", n_jobs=-1, verbose=2)
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Best params: {best_params}")
    print(f"Accuracy on Test: {accuracy}\n")


GridSearch for Logistic Regression
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best params: {'C': 0.1}
Accuracy on Test: 0.6586886063325854

GridSearch for Decision Tree
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best params: {'max_depth': 5}
Accuracy on Test: 0.6584392919471453

GridSearch for Random Forest
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best params: {'max_depth': 20, 'n_estimators': 100}
Accuracy on Test: 0.6599351782597855

GridSearch for Gradient Boosting
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best params: {'learning_rate': 0.05, 'n_estimators': 100}
Accuracy on Test: 0.6589379207180255

GridSearch for Support Vector Classifier
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best params: {'C': 1.0}
Accuracy on Test: 0.6619296933433059



In [20]:
most_common_class = Counter(y_train).most_common(1)[0][0]
y_pred_naive = np.full_like(y_test, most_common_class)
naive_accuracy = accuracy_score(y_test, y_pred_naive)

print(f"Accuracy of Naive Classifier: {naive_accuracy}")

Accuracy of Naive Classifier: 0.6576913487908252


#### Classification for bad, so-so, great + f1 score

In [21]:
def binarize_target(rating):
    if rating <= 1:
        return 'bad'
    elif rating <= 3:
        return "so-so"
    else:
        return "great"

In [22]:
y = y.apply(binarize_target)

In [23]:
y.value_counts()

rating
great    17396
bad       1836
so-so      820
Name: count, dtype: int64

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [25]:

models = {
    "Logistic Regression": (LogisticRegression(max_iter=5000), {"C": [0.1, 1, 10]}),
    "Decision Tree": (DecisionTreeClassifier(), {"max_depth": [5, 10, 20]}),
    "Random Forest": (RandomForestClassifier(), {"n_estimators": [100, 200], "max_depth": [10, 20]}),
    "Gradient Boosting": (GradientBoostingClassifier(), {"n_estimators": [50, 100], "learning_rate": [0.05, 0.1]}),
    "Support Vector Classifier": (SVC(), {"C": [0.1, 1.0]}),
}

best_accuracy = 0
best_class_model = None

for model_name, (model, params) in models.items():
    print(f"GridSearch for {model_name}")
    
    grid_search = GridSearchCV(model, params, cv=skf, scoring="accuracy", n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Best params: {best_params}")
    print(f"Accuracy on Test: {accuracy}\n")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_class_model = best_model


GridSearch for Logistic Regression
Best params: {'C': 0.1}
Accuracy on Test: 0.8681126901022189

GridSearch for Decision Tree
Best params: {'max_depth': 5}
Accuracy on Test: 0.8666168037895786

GridSearch for Random Forest
Best params: {'max_depth': 20, 'n_estimators': 200}
Accuracy on Test: 0.868611318873099

GridSearch for Gradient Boosting
Best params: {'learning_rate': 0.1, 'n_estimators': 50}
Accuracy on Test: 0.8676140613313388

GridSearch for Support Vector Classifier
Best params: {'C': 1.0}
Accuracy on Test: 0.8681126901022189



In [26]:
most_common_class = y_train.value_counts().idxmax()

y_pred_dummy = [most_common_class] * len(y_test)

naive_accuracy = accuracy_score(y_test, y_pred_dummy)

print(f"Accuracy of naive classifier (predicting '{most_common_class}'): {naive_accuracy:.4f}")

Accuracy of naive classifier (predicting 'great'): 0.8676


In [27]:
test_accuracy = accuracy_score(y_test, best_class_model.predict(X_test))
print(f"Best Model: {best_class_model.__class__.__name__}, Accuracy: {test_accuracy}")

Best Model: RandomForestClassifier, Accuracy: 0.868611318873099


In [28]:
y_pred = best_class_model.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

         bad       0.64      0.02      0.04       367
       great       0.87      1.00      0.93      3480
       so-so       1.00      0.01      0.01       164

    accuracy                           0.87      4011
   macro avg       0.84      0.34      0.33      4011
weighted avg       0.85      0.87      0.81      4011

Confusion Matrix:
[[   7  360    0]
 [   4 3476    0]
 [   0  163    1]]


In [29]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
xgb_model = xgb.XGBClassifier(eval_metric="mlogloss") 

params_xgb = {
    "n_estimators": [50, 100],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 6],
}

grid_search_xgb = GridSearchCV(xgb_model, params_xgb, cv=skf, scoring="accuracy", n_jobs=-1)
grid_search_xgb.fit(X_train, y_train_encoded)

best_xgb_model = grid_search_xgb.best_estimator_
best_xgb_params = grid_search_xgb.best_params_

y_pred_xgb_encoded = best_xgb_model.predict(X_test)
y_pred_xgb = label_encoder.inverse_transform(y_pred_xgb_encoded)  

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print(f"Best params: {best_xgb_params}")
print(f"Accuracy on Test: {accuracy_xgb}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_xgb, zero_division=1))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100}
Accuracy on Test: 0.8676140613313388

Classification Report:
              precision    recall  f1-score   support

         bad       0.50      0.01      0.01       367
       great       0.87      1.00      0.93      3480
       so-so       1.00      0.00      0.00       164

    accuracy                           0.87      4011
   macro avg       0.79      0.33      0.31      4011
weighted avg       0.84      0.87      0.81      4011

Confusion Matrix:
[[   2  365    0]
 [   2 3478    0]
 [   0  164    0]]


Class imbalance — there are too many examples of "great" (3480) compared to "bad" (367) and **"so-so" (164).
The model mainly learns the "great" class and does not learn the small classes well.

XGBoost minimizes log loss. When the model is uncertain, it prefers to predict the most frequent class ("great") instead of risking a rare class prediction.**

#### SMOTE (Synthetic Minority Over-sampling Technique)
SMOTE works by creating new synthetic examples of the minority class based on nearby existing samples.
Instead of simply duplicating rare examples, it generates new points between neighbors in feature space.

This helps make the class distribution more balanced while preserving the data structure and patterns.

In [31]:
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test) 

smote = SMOTE(sampling_strategy="auto", random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train_encoded)

xgb_classifier = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="mlogloss",
)
xgb_classifier.fit(X_train_balanced, y_train_balanced)

y_pred_encoded = xgb_classifier.predict(X_test)

y_pred = label_encoder.inverse_transform(y_pred_encoded)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on Test: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy on Test: 0.6731

Classification Report:
              precision    recall  f1-score   support

         bad       0.22      0.53      0.31       367
       great       0.91      0.71      0.80      3480
       so-so       0.06      0.16      0.09       164

    accuracy                           0.67      4011
   macro avg       0.40      0.47      0.40      4011
weighted avg       0.81      0.67      0.73      4011


Confusion Matrix:
[[ 194  142   31]
 [ 652 2480  348]
 [  39   99   26]]


The model performance is still not fully satisfactory, but some metrics have improved.

### Class "bad"
- **Precision = 0.21**: Only 21% of the cases predicted as *"bad"* are actually bad.  
- **Recall = 0.51**: The model correctly identifies 53% of all true *"bad"* samples.  
- **F1-score = 0.30**: The low F1-score indicates an imbalance between precision and recall.

### Class "great"
- **Precision = 0.91**: When the model predicts *"great"*, it is correct in 91% of cases.  
- **Recall = 0.71**: The model detects 71% of all true *"great"* samples.  
- **F1-score = 0.80**: This reflects a good balance between precision and recall.

### Class "so-so"
The model is still almost unable to recognize the *"so-so"* class, which results in a very low F1-score (**0.09**).

### Conclusion
The classifier remains biased toward the majority class *"great"*.  
Although recall for *"bad"* has improved, precision is still low.  
The *"so-so"* class suffers the most due to strong class imbalance and overlapping feature distributions.


Before applying SMOTE, the model was trained on a highly imbalanced dataset where the **"great"** class dominated.  
As a result, the model almost always predicted **"great"**, which produced high accuracy but poor performance (low F1-score) for the **"bad"** and **"so-so"** classes.


In [32]:
joblib.dump(xgb_classifier, "data/best_classification_model.pkl")

['data/best_classification_model.pkl']

#### Nutrition Facts

In [33]:
api_key = "Kf9B6CO8znfPSHJ5p1xfiEEKLZi3OY3BakkjGsvj"
nutrition_url = "https://api.nal.usda.gov/fdc/v1/foods/search"

nutrition_data = []

daily = pd.read_csv("data/daily.tsv", sep="\t", converters={"Daily Value": lambda value: float(value.replace(",", ""))})
for ingr in tqdm(ingredients):
    if "/" in ingr:
        ingr_req = ingr.split("/")
    else:
        ingr_req = [ingr]
        
    for ingredient in ingr_req:
        response = requests.get(nutrition_url, params={
            "query": ingredient,
            "api_key": api_key,
            "pageSize": 1
        })
        if response.status_code == 200:
            nutrition_info = response.json()
            if nutrition_info['totalHits'] == 0:
                continue
            nutrients = nutrition_info["foods"][0]["foodNutrients"]
            nutrient_dict = {"ingredient": ingredient}
            for nutrient in nutrients:
                for _, row in daily.iterrows():
                    if row["Nutrient"].lower() in nutrient["nutrientName"].lower():
                        nutrient_dict[row["Nutrient"]] = round(nutrient["value"] / row["Daily Value"] * 100)
                        break

            nutrition_data.append(nutrient_dict)

100%|██████████| 231/231 [04:18<00:00,  1.12s/it]


In [34]:
nutrition_df = pd.DataFrame(nutrition_data)
nutrition_df.to_csv("data/nutrition_facts.csv", index=False)

In [35]:
df = pd.read_csv("data/epi_r.csv")
print(df.columns)


Index(['title', 'rating', 'calories', 'protein', 'fat', 'sodium', '#cakeweek',
       '#wasteless', '22-minute meals', '3-ingredient recipes',
       ...
       'yellow squash', 'yogurt', 'yonkers', 'yuca', 'zucchini', 'cookbooks',
       'leftovers', 'snack', 'snack week', 'turkey'],
      dtype='object', length=680)


In [36]:
recipe_urls = []
for _, row in df.iterrows():
    if pd.notna(row['title']):
        recipe_urls.append({
            "title": row["title"],
            "rating": row["rating"],
            "url": f"https://www.epicurious.com/recipes/food/views/{row['title'].replace(' ', '-').lower()}"
        })

In [37]:
recipe_urls_df = pd.DataFrame(recipe_urls)
recipe_urls_df.to_csv("data/similar_recipes.csv", index=False)